# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library.

### Dataset Source
The dataset is described using a Croissant schema, accessible at the URL below.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. We start by providing the URL to the Croissant schema and loading the metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print the dataset title and description
print(f"{metadata.name}: {metadata.description}")

# Print when the dataset was published
print(f"Date published: {getattr(metadata, 'datePublished', 'Unknown')}")

## 2. Data Overview
Review available record sets and fields. Each entity in the schema has a globally unique `@id`.

Let's list all available record sets, their `@id`, and main fields (by `@id`).

In [ ]:
# List all record sets by their @id and print contained fields

pp = pprint.PrettyPrinter(indent=2)

record_sets = [rs for rs in dataset.metadata.record_sets]
print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"  Record set name: {rs.name}")
    print(f"    @id: {rs.id}")
    field_ids = [f.id for f in getattr(rs, 'fields', [])]
    print(f"    Fields (@id):\n      {field_ids}")
    print("")

# For demonstration, print the full field details for the first record set:
if record_sets:
    first_rs = record_sets[0]
    print(f"Sample fields (@id and name) for record set '{first_rs.name}' ({first_rs.id}):")
    for f in getattr(first_rs, 'fields', []):
        print(f"    {f.id} : {f.name}")

## 3. Data Extraction
Convert the records from key record sets to Pandas DataFrames. All lookups and operations use entity `@id` fields.

Below, we extract all tabular survey data for analysis.

In [ ]:
# Gather the @id values of all record sets
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print("No records found.")

# Pick the main survey record set for further analysis (select the largest by row count)
if dataframes:
    main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k]))
    print(f"Selected main record set: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's filter the survey data by a numeric field, normalize it, and group by another (e.g., education or gender). All fields are referenced by their `@id`.

_Check the schema field names/IDs to adjust the fields as needed for your dataset analysis._

In [ ]:
# Choose a numeric field and a grouping field by their @id.
# (Refer to the previous output for available columns and adjust accordingly.)

survey_df = dataframes[main_record_set_id]
print("Available columns in main survey DataFrame:")
print(survey_df.columns.tolist())

# Example: GAD-7 overall score field (adjust as by field @id, e.g., 'gad7_total' or similar)
# Use actual column name/@id from your loaded DataFrame (example below uses a placeholder)
numeric_field = None
possible_gad7_fields = [c for c in survey_df.columns if 'gad' in c.lower() and 'total' in c.lower()]
if possible_gad7_fields:
    numeric_field = possible_gad7_fields[0]
else:
    # Fallback to first numeric field
    numeric_candidates = survey_df.select_dtypes(include='number').columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]

if numeric_field is not None:
    print(f"Selected numeric field for demonstration: {numeric_field}")
    threshold = 10
    filtered_df = survey_df[survey_df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold} (n={len(filtered_df)}):")
    print(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Now group by a field (e.g., education or gender)
    group_field_candidates = [c for c in survey_df.columns if ('education' in c.lower() or 'gender' in c.lower())]
    group_field = None
    if group_field_candidates:
        group_field = group_field_candidates[0]
        print(f"Grouping by {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Mean {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for demonstration.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its values across different groups. All visualizations are based on `@id` references in the DataFrame.

_Below is an example using matplotlib/seaborn; you can adapt as needed by changing the field IDs according to your actual field names._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(survey_df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    if group_field is not None:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=survey_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to use the `mlcroissant` library to:
1. Load and inspect Croissant-based metadata for a research dataset,
2. View available record sets and their field IDs,
3. Extract all survey records and organize them into DataFrames by record set `@id`,
4. Select, filter, normalize, and group data using field `@id`,
5. Visualize numeric field distributions and groupings.

All selections in the notebook are compatible with the Croissant schema, and every entity is referenced via its `@id` consistently.